# Smart-Demand: Category-Level Sales Prediction

**Machine Learning | Final Project**

**Team:**
- Christopher Setyawan (2802459670)
- Klaus Anson (2802459361)
- Owen Aldrich Setiawan (2802463314)
- Lawrance Velasques (2802477080)
- Gracias Kumara Winata (2802459683)

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Task:** Regression: predict total units sold per product category per month  
**Main Model:** Random Forest Regressor  
**Baseline:** Linear Regression

## 1. Setup & Library Import

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')

import sklearn
print(f'pandas       : {pd.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print('Libraries loaded.')

## 2. Load Dataset

We use 6 out of 9 CSV files from the Olist dataset. The geolocation, customers, and sellers tables are not used since our prediction focuses on category-level sales, not location analysis.

Make sure all 6 CSV files are already uploaded to the shared Google Drive folder before running the cells below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH  = '/content/drive/MyDrive/dataset/'
MODEL_PATH = '/content/drive/MyDrive/SmartDemand_Dataset/models/'
os.makedirs(MODEL_PATH, exist_ok=True)
print('Drive mounted.')

In [ ]:
df_orders     = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
df_items      = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
df_products   = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
df_reviews    = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
df_payments   = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
df_cat_transl = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('Loaded:')
for name, df in [('orders',df_orders),('items',df_items),
                  ('products',df_products),('reviews',df_reviews),
                  ('payments',df_payments),('cat_transl',df_cat_transl)]:
    print(f'  {name:12s}: {df.shape}')

## 3. Data Integration

All 6 tables are joined into one master dataset, then aggregated at the category-month level. Each row in the final dataset represents one product category in one specific month.

Join order: items + orders + reviews + payments + products + category translation, then filter delivered orders only, then aggregate by category and month.

In [ ]:
# Aggregate reviews per order
review_avg = (
    df_reviews
    .groupby('order_id')['review_score']
    .mean().reset_index()
    .rename(columns={'review_score': 'product_rating'})
)

# Aggregate payments per order
payment_per_order = (
    df_payments
    .groupby('order_id')['payment_value']
    .sum().reset_index()
    .rename(columns={'payment_value': 'total_payment'})
)

# Build master dataset
master = df_items.copy()
master = master.merge(
    df_orders[['order_id','order_purchase_timestamp','order_status']],
    on='order_id', how='left'
)
master = master.merge(review_avg, on='order_id', how='left')
master = master.merge(payment_per_order, on='order_id', how='left')
master = master.merge(
    df_products[['product_id','product_category_name']],
    on='product_id', how='left'
)
master = master.merge(df_cat_transl, on='product_category_name', how='left')

# Use English category name
master['category'] = (
    master['product_category_name_english']
    .fillna(master['product_category_name'])
    .fillna('unknown')
)

# Filter delivered orders only
master = master[master['order_status'] == 'delivered'].copy()

# Parse dates
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
master['year']  = master['order_purchase_timestamp'].dt.year
master['month'] = master['order_purchase_timestamp'].dt.month

# Item total per order for discount calculation
order_item_total = (
    master.groupby('order_id')['price']
    .sum().reset_index()
    .rename(columns={'price': 'item_total'})
)
master = master.merge(order_item_total, on='order_id', how='left')

print(f'Master dataset shape: {master.shape}')
print(f'Date range: {master["year"].min()} – {master["year"].max()}')

In [ ]:
# Aggregate to category-month level
df_agg = (
    master
    .groupby(['category', 'year', 'month'])
    .agg(
        quantity_sold   = ('order_item_id', 'count'),
        avg_price       = ('price', 'mean'),
        avg_freight     = ('freight_value', 'mean'),
        avg_rating      = ('product_rating', 'mean'),
        avg_item_total  = ('item_total', 'mean'),
        avg_payment     = ('total_payment', 'mean'),
        num_products    = ('product_id', 'nunique'),       # product variety in category
        min_price       = ('price', 'min'),
        max_price       = ('price', 'max'),
    )
    .reset_index()
)

print(f'Category-level dataset shape: {df_agg.shape}')
print(f'Unique categories: {df_agg["category"].nunique()}')
print(f'\nSample quantity_sold range:')
print(f'  Min    : {df_agg["quantity_sold"].min()}')
print(f'  Median : {df_agg["quantity_sold"].median():.0f}')
print(f'  Max    : {df_agg["quantity_sold"].max()}')
display(df_agg.head(5))

## 4. Data Cleaning

In [ ]:
df_clean = df_agg.copy()

# Fill missing ratings with overall median
df_clean['avg_rating'] = df_clean['avg_rating'].fillna(df_clean['avg_rating'].median())

# Fill missing payment with item total
df_clean['avg_payment'] = df_clean['avg_payment'].fillna(df_clean['avg_item_total'])

# Drop rows with missing price or freight
df_clean = df_clean.dropna(subset=['avg_price', 'avg_freight'])

# Remove extreme outliers (3x IQR)
q1  = df_clean['quantity_sold'].quantile(0.25)
q3  = df_clean['quantity_sold'].quantile(0.75)
iqr = q3 - q1
df_clean = df_clean[df_clean['quantity_sold'] <= q3 + 3 * iqr]

print(f'Rows after cleaning : {len(df_clean):,}')
print(f'Missing values      : {df_clean.isnull().sum().sum()}')
print(f'\nQuantity sold stats:')
print(df_clean['quantity_sold'].describe().round(1))

## 5. Feature Engineering

Features used in this model:

- avg_price - average selling price in the category this month
- discount_rate - estimated discount ratio based on payment vs item total
- avg_freight - average shipping cost in the category
- avg_rating - average customer review score in the category
- num_products - number of unique products, reflects category variety
- price_range - difference between max and min price in the category
- category_encoded - label-encoded category name
- seasonality_index - normalized monthly demand index (0 = slowest, 1 = busiest)
- last_month_sales - total units sold in this category last month (lag-1)

In [ ]:
df_feat = df_clean.copy()

df_feat['discount_rate'] = (
    (df_feat['avg_item_total'] - df_feat['avg_payment'])
    / df_feat['avg_item_total'].replace(0, np.nan)
).clip(lower=0).fillna(0)

df_feat['price_range'] = df_feat['max_price'] - df_feat['min_price']

monthly_avg  = df_feat.groupby('month')['quantity_sold'].mean()
monthly_norm = (monthly_avg - monthly_avg.min()) / (monthly_avg.max() - monthly_avg.min())
df_feat['seasonality_index'] = df_feat['month'].map(monthly_norm)

df_feat = df_feat.sort_values(['category', 'year', 'month'])
df_feat['last_month_sales'] = (
    df_feat.groupby('category')['quantity_sold']
    .shift(1).fillna(0)
)

label_enc = LabelEncoder()
df_feat['category_encoded'] = label_enc.fit_transform(df_feat['category'])

print('Feature engineering complete.')

In [ ]:
FEATURES = [
    'avg_price',
    'discount_rate',
    'avg_freight',
    'avg_rating',
    'num_products',
    'price_range',
    'category_encoded',
    'seasonality_index',
    'last_month_sales',
]
TARGET = 'quantity_sold'

print(f'Features ({len(FEATURES)}) : {FEATURES}')
print(f'Target              : {TARGET}')
print(f'Dataset shape       : {df_feat[FEATURES + [TARGET]].shape}')
display(df_feat[FEATURES + [TARGET]].describe().round(2))

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# 6.1 Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_feat[TARGET], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Quantity Sold (Category-Level)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Units Sold per Category per Month')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df_feat[TARGET]), bins=50, color='cornflowerblue', edgecolor='white')
axes[1].set_title('log(Quantity Sold + 1)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('log(Units + 1)')
axes[1].set_ylabel('Frequency')

plt.suptitle('Target Variable — Category-Level Sales', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_01_target_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Skewness : {df_feat[TARGET].skew():.3f}')
print(f'Mean     : {df_feat[TARGET].mean():.1f}')
print(f'Median   : {df_feat[TARGET].median():.1f}')
print(f'Max      : {df_feat[TARGET].max():.0f}')

In [ ]:
# 6.2 Correlation heatmap
corr_matrix = df_feat[FEATURES + [TARGET]].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='Blues', center=0, linewidths=0.5, annot_kws={'size': 9}
)
plt.title('Correlation Heatmap — Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_02_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

print('Correlation with quantity_sold:')
corr_target = corr_matrix[TARGET].drop(TARGET).sort_values(ascending=False)
print(corr_target.round(4).to_string())

In [ ]:
# 6.3 Top 15 categories by total sales
top_cats = (
    df_feat.groupby('category')['quantity_sold']
    .sum().sort_values(ascending=False).head(15)
)

plt.figure(figsize=(12, 6))
colors = plt.cm.Blues(np.linspace(0.35, 0.85, 15))
plt.barh(top_cats.index[::-1], top_cats.values[::-1], color=colors[::-1])
plt.title('Top 15 Categories by Total Sales Volume', fontsize=13, fontweight='bold')
plt.xlabel('Total Quantity Sold')
plt.tight_layout()
plt.savefig('eda_03_top_categories.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.4 Monthly sales trend
monthly_sales = (
    df_feat.groupby(['year','month'])['quantity_sold']
    .sum().reset_index()
)
monthly_sales['period'] = pd.to_datetime(
    monthly_sales.assign(day=1)[['year','month','day']]
)

plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['period'], monthly_sales['quantity_sold'],
         marker='o', color='steelblue', linewidth=2, markersize=5)
plt.fill_between(monthly_sales['period'], monthly_sales['quantity_sold'],
                 alpha=0.15, color='steelblue')
plt.title('Monthly Sales Trend — All Categories Combined', fontsize=13, fontweight='bold')
plt.xlabel('Period')
plt.ylabel('Total Units Sold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('eda_04_monthly_trend.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 6.5 Last month sales vs current sales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_feat['last_month_sales'], df_feat['quantity_sold'],
                alpha=0.4, color='steelblue', s=20)
axes[0].set_title('Last Month Sales vs Current Month Sales', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Last Month Sales')
axes[0].set_ylabel('Current Month Sales')

axes[1].scatter(df_feat['num_products'], df_feat['quantity_sold'],
                alpha=0.4, color='cornflowerblue', s=20)
axes[1].set_title('Product Variety vs Sales Volume', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Unique Products in Category')
axes[1].set_ylabel('Total Units Sold')

plt.tight_layout()
plt.savefig('eda_05_feature_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Train-Test Split

In [ ]:
X = df_feat[FEATURES].values
y = df_feat[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train : {X_train.shape[0]:,} rows ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]:,} rows ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nTarget stats (category-level):')
print(f'  Mean   : {y.mean():.1f} units per category per month')
print(f'  Median : {np.median(y):.1f} units')
print(f'  Max    : {y.max():.0f} units')

## 8. Model Training

In [ ]:
print('Training Linear Regression (baseline)...')
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print('Done.')

In [ ]:
print('Training Random Forest Regressor...')
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('Done.')

## 9. Evaluation & Comparison

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test, name):
    train_pred = model.predict(X_train)
    test_pred  = model.predict(X_test)
    results = {
        'Model'      : name,
        'Train MAE'  : mean_absolute_error(y_train, train_pred),
        'Test MAE'   : mean_absolute_error(y_test, test_pred),
        'Train RMSE' : np.sqrt(mean_squared_error(y_train, train_pred)),
        'Test RMSE'  : np.sqrt(mean_squared_error(y_test, test_pred)),
        'Train R2'   : r2_score(y_train, train_pred),
        'Test R2'    : r2_score(y_test, test_pred),
    }
    return results, test_pred

lr_results, lr_pred = evaluate(lr_model, X_train, X_test, y_train, y_test, 'Linear Regression')
rf_results, rf_pred = evaluate(rf_model, X_train, X_test, y_train, y_test, 'Random Forest')

results_df = pd.DataFrame([lr_results, rf_results]).set_index('Model')
print('Model Comparison:')
display(results_df.round(4))

print(f'\nRandom Forest Test R2  : {rf_results["Test R2"]:.4f}')
print(f'Linear Regression R2   : {lr_results["Test R2"]:.4f}')

In [ ]:
gap = rf_results['Train R2'] - rf_results['Test R2']
print(f'Train R2 : {rf_results["Train R2"]:.4f}')
print(f'Test R2  : {rf_results["Test R2"]:.4f}')
print(f'Gap      : {gap:.4f}')
if gap < 0.05:
    print('Status   : No overfitting detected.')
elif gap < 0.15:
    print('Status   : Slight overfitting — acceptable.')
else:
    print('Status   : Overfitting detected.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred, name, color in zip(
    axes, [lr_pred, rf_pred],
    ['Linear Regression (Baseline)', 'Random Forest'],
    ['tomato', 'steelblue']
):
    ax.scatter(y_test, pred, alpha=0.4, color=color, s=15)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f'{name}\nR2 = {r2_score(y_test, pred):.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Actual Units Sold')
    ax.set_ylabel('Predicted Units Sold')
    ax.legend()

plt.suptitle('Actual vs Predicted — Category-Level Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eval_actual_vs_predicted.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
residuals = y_test - rf_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(rf_pred, residuals, alpha=0.3, color='steelblue', s=15)
axes[0].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[0].set_title('Residual Plot — Random Forest', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Residual Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('eval_residuals.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'feature'    : FEATURES,
    'importance' : rf_model.feature_importances_
}).sort_values('importance', ascending=True)

plt.figure(figsize=(10, 6))
bar_colors = plt.cm.Blues(np.linspace(0.3, 0.85, len(FEATURES)))
plt.barh(importance_df['feature'], importance_df['importance'], color=bar_colors)
plt.title('Feature Importance — Random Forest (Category-Level)', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print('Feature importance (highest to lowest):')
display(importance_df.sort_values('importance', ascending=False).round(4))

## 11. Save Model

All model files are saved to the shared Google Drive folder and will be loaded by the deployment app.

In [ ]:
joblib.dump(rf_model,  MODEL_PATH + 'random_forest_model.joblib')
joblib.dump(lr_model,  MODEL_PATH + 'linear_regression_model.joblib')
joblib.dump(label_enc, MODEL_PATH + 'label_encoder.joblib')

config = {
    'features'          : FEATURES,
    'target'            : TARGET,
    'granularity'       : 'category_month',
    'categories'        : list(label_enc.classes_),
    'seasonality_index' : {int(k): float(v) for k, v in monthly_norm.items()},
    'log_transform'     : False,
    'target_stats': {
        'mean'  : round(float(y.mean()), 2),
        'median': round(float(np.median(y)), 2),
        'max'   : round(float(y.max()), 2),
    },
    'model_metrics': {
        'random_forest': {
            'test_mae'  : round(float(rf_results['Test MAE']), 4),
            'test_rmse' : round(float(rf_results['Test RMSE']), 4),
            'test_r2'   : round(float(rf_results['Test R2']), 4)
        },
        'linear_regression': {
            'test_mae'  : round(float(lr_results['Test MAE']), 4),
            'test_rmse' : round(float(lr_results['Test RMSE']), 4),
            'test_r2'   : round(float(lr_results['Test R2']), 4)
        }
    }
}

with open(MODEL_PATH + 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved to:', MODEL_PATH)
print('  random_forest_model.joblib')
print('  linear_regression_model.joblib')
print('  label_encoder.joblib')
print('  config.json')
print(f'\nFinal Random Forest Test R2 : {rf_results["Test R2"]:.4f}')
print(f'Linear Regression Test R2   : {lr_results["Test R2"]:.4f}')